# DistilBERT Fine-tuning — AI NewsOps Platform
**AIA Bloc 4 — MLOps Pipeline · Lightning AI Studio**

⚡ Avant de lancer : sélectionner un GPU dans la barre du haut du Studio (T4, L4 ou A10G).
Pas besoin d'upload manuel — le filesystem Lightning est persistant, tout reste après fermeture.

In [ ]:
#!pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.1/532.1 MB 52.5 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 65.2 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 87.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 55.0 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 71.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.6/197.6 MB 74.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 89.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 53.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 73.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 87.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/

In [1]:
# ── Cellule 1 : Vérification GPU ─────────────────────────────
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Aucun GPU détecté — vérifie le sélecteur de machine en haut du Studio Lightning")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Prêt sur : {DEVICE}")

GPU disponible : False
⚠️  Aucun GPU détecté — vérifie le sélecteur de machine en haut du Studio Lightning

✅ Prêt sur : cpu


In [ ]:
# ── Cellule 2 : Installation des dépendances ─────────────────
# Lightning Studios ont souvent déjà torch préinstallé — on installe le reste
#!pip install -q transformers kagglehub mlflow pyarrow scikit-learn matplotlib seaborn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.2.2 requires a2wsgi>=1.10.8, which is not installed.
apache-airflow-core 3.2.2 requires argcomplete>=1.10, which is not installed.
apache-airflow-core 3.2.2 requires asgiref>=2.3.0; python_version < "3.14", which is not installed.
apache-airflow-core 3.2.2 requires cron-descriptor>=1.2.24, which is not installed.
apache-airflow-core 3.2.2 requires croniter>=2.0.2, which is not installed.
apache-airflow-core 3.2.2 requires deprecated>=1.2.13, which is not installed.
apache-airflow-core 3.2.2 requires dill>=0.2.2, which is not installed.
apache-airflow-core 3.2.2 requires jsonschema>=4.19.1, which is not installed.
apache-airflow-core 3.2.2 requires lazy-object-proxy>=1.2.0, which is not installed.
apache-airflow-core 3.2.2 requires linkify-it-py>=2.0.0, which is not installed.
apache-airflow-c

In [2]:
# ── Cellule 3 : Config Kaggle ────────────────────────────────
# Sur Lightning, pas d'upload navigateur : on écrit le token directement
# Récupère ton token sur https://www.kaggle.com/settings → API → Create New Token
# Colle ton username et ta clé ci-dessous (remplace les valeurs), ou monte le
# fichier kaggle.json directement dans le panneau Files du Studio sous ~/.kaggle/

import os
from pathlib import Path

KAGGLE_USERNAME = "TON_USERNAME_KAGGLE"   # ← à remplacer
KAGGLE_KEY      = "TA_CLE_API_KAGGLE"     # ← à remplacer

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    import json as _json
    kaggle_json.write_text(_json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
    os.chmod(kaggle_json, 0o600)
    print("✅ kaggle.json créé")
else:
    print("✅ kaggle.json déjà présent (filesystem persistant Lightning)")

✅ kaggle.json déjà présent (filesystem persistant Lightning)


In [3]:
# ── Cellule 4 : Téléchargement & Preprocessing ───────────────
import re, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import kagglehub

warnings.filterwarnings('ignore')

# Dossier de travail persistant Lightning — survit aux redémarrages du Studio
WORKDIR = Path.home() / "newsops-artifacts"
WORKDIR.mkdir(exist_ok=True)

CATEGORY_MAPPING = {
    'POLITICS': 'politics', 'THE WORLDPOST': 'politics', 'WORLD NEWS': 'politics',
    'WORLDPOST': 'politics', 'U.S. NEWS': 'politics',
    'WELLNESS': 'health_wellness', 'HEALTHY LIVING': 'health_wellness', 'TASTE': 'health_wellness',
    'ENTERTAINMENT': 'entertainment', 'COMEDY': 'entertainment', 'WEIRD NEWS': 'entertainment',
    'TRAVEL': 'lifestyle', 'STYLE & BEAUTY': 'lifestyle', 'FOOD & DRINK': 'lifestyle',
    'HOME & LIVING': 'lifestyle', 'GOOD NEWS': 'lifestyle', 'STYLE': 'lifestyle',
    'PARENTING': 'family_education', 'PARENTS': 'family_education',
    'COLLEGE': 'family_education', 'EDUCATION': 'family_education',
    'QUEER VOICES': 'media', 'BLACK VOICES': 'media', 'WOMEN': 'media',
    'MEDIA': 'media', 'LATINO VOICES': 'media',
    'BUSINESS': 'business', 'MONEY': 'business', 'FIFTY': 'business',
    'SPORTS': 'sports',
    'IMPACT': 'international', 'RELIGION': 'international',
    'GREEN': 'tech_science', 'SCIENCE': 'tech_science',
    'TECH': 'tech_science', 'ENVIRONMENT': 'tech_science',
    'ARTS': 'arts_culture', 'ARTS & CULTURE': 'arts_culture', 'CULTURE & ARTS': 'arts_culture',
    'CRIME': 'crime',
    'WEDDINGS': 'other', 'DIVORCE': 'other',
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s\-\']', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# Cache : si déjà preprocessé sur ce Studio, on ne refait pas le travail
PROCESSED_FLAG = WORKDIR / "train.parquet"

if PROCESSED_FLAG.exists():
    print("✅ Données déjà preprocessées trouvées sur le disque persistant — chargement direct")
    train_df = pd.read_parquet(WORKDIR / "train.parquet")
    val_df   = pd.read_parquet(WORKDIR / "val.parquet")
    test_df  = pd.read_parquet(WORKDIR / "test.parquet")
    with open(WORKDIR / "label_mapping.json") as f:
        id2label = {int(k): v for k, v in json.load(f).items()}
else:
    print("Téléchargement du dataset depuis Kaggle...")
    import os
    path = kagglehub.dataset_download('rmisra/news-category-dataset')
    JSON_PATH = os.path.join(path, 'News_Category_Dataset_v3.json')

    df = pd.read_json(JSON_PATH, lines=True)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.drop_duplicates(subset=['headline', 'short_description'])
    df = df.dropna(subset=['headline'])
    df['short_description'] = df['short_description'].fillna('')
    df = df[(df['date'].dt.year >= 2012) & (df['date'].dt.year <= 2022)]
    df = df[df['headline'].str.len() >= 10].reset_index(drop=True)

    df['category'] = df['category'].map(CATEGORY_MAPPING).fillna('other')
    df['text'] = df['headline'] + ' [SEP] ' + df['short_description']
    df['text_clean'] = df['text'].apply(clean_text)

    le = LabelEncoder()
    df['label'] = le.fit_transform(df['category'])
    id2label = {int(i): cls for i, cls in enumerate(le.classes_)}

    train_df, temp = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp, test_size=0.50, random_state=42, stratify=temp['label'])

    train_df.to_parquet(WORKDIR / "train.parquet")
    val_df.to_parquet(WORKDIR / "val.parquet")
    test_df.to_parquet(WORKDIR / "test.parquet")
    with open(WORKDIR / "label_mapping.json", "w") as f:
        json.dump(id2label, f, indent=2)
    print("✅ Preprocessing terminé et sauvegardé sur le disque persistant")

label2id = {v: k for k, v in id2label.items()}
NUM_LABELS = len(id2label)
CLASS_NAMES = [id2label[i] for i in sorted(id2label.keys())]

print(f"\n✅ Dataset prêt")
print(f"   Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"   Classes ({NUM_LABELS}) : {CLASS_NAMES}")

✅ Données déjà preprocessées trouvées sur le disque persistant — chargement direct

✅ Dataset prêt
   Train : 146,113 | Val : 31,310 | Test : 31,310
   Classes (13) : ['arts_culture', 'business', 'crime', 'entertainment', 'family_education', 'health_wellness', 'international', 'lifestyle', 'media', 'other', 'politics', 'sports', 'tech_science']


In [4]:
# ── Cellule 5 : Dataset PyTorch (pré-tokenisé en RAM) ────────
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast

MAX_LENGTH = 128
BATCH_SIZE = 64 if torch.cuda.is_available() else 16
MODEL_NAME = 'distilbert-base-uncased'

class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        texts  = df['text'].tolist()
        labels = df['label'].tolist()
        enc = tokenizer(texts, max_length=max_length, padding='max_length',
                        truncation=True, return_tensors='pt')
        self.input_ids      = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'label':          self.labels[idx],
        }

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

print("Pré-tokenisation...")
train_ds = NewsDataset(train_df, tokenizer)
val_ds   = NewsDataset(val_df,   tokenizer)
test_ds  = NewsDataset(test_df,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print(f"✅ Dataloaders prêts — batch_size={BATCH_SIZE} | steps/epoch={len(train_loader):,}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Pré-tokenisation...
✅ Dataloaders prêts — batch_size=16 | steps/epoch=9,133


In [5]:
# ── Cellule 6 : Modèle + Optimizer (anti-overfitting) ────────
# Fix the missing bz2 runtime library required by transformers
!apt-get update -y
!apt-get install -y bzip2 libbz2-dev
%pip install -U transformers

from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

EPOCHS        = 5      # avec early stopping — s'arrêtera avant si besoin
LEARNING_RATE = 3e-5
PATIENCE      = 2      # arrêt anticipé si pas d'amélioration sur 2 epochs
WEIGHT_DECAY  = 0.05   # régularisation renforcée
DROPOUT       = 0.2    # dropout natif DistilBERT augmenté (défaut 0.1)

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    dropout=DROPOUT,
    seq_classif_dropout=DROPOUT,
).to(DEVICE)

optimizer     = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.06 * total_steps)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

total_params  = sum(p.numel() for p in model.parameters())
print(f"✅ DistilBERT chargé (dropout={DROPOUT}, weight_decay={WEIGHT_DECAY})")
print(f"   Paramètres  : {total_params:,}")
print(f"   Total steps : {total_steps:,} (max) | Warmup : {warmup_steps:,}")
print(f"   Early stopping patience : {PATIENCE} epochs")

/bin/bash: ligne 1: apt-get: commande introuvable
/bin/bash: ligne 1: apt-get: commande introuvable
Note: you may need to restart the kernel to use updated packages.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DistilBERT chargé (dropout=0.2, weight_decay=0.05)
   Paramètres  : 66,963,469
   Total steps : 45,665 (max) | Warmup : 2,739
   Early stopping patience : 2 epochs


In [6]:
# ── Cellule 7 : Boucle d'entraînement avec early stopping ────
from sklearn.metrics import f1_score, accuracy_score, classification_report
import time

CHECKPOINT_DIR = WORKDIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
BEST_MODEL_DIR = WORKDIR / "best_model"

def train_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for i, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()
        if (i + 1) % 200 == 0:
            print(f"  Step {i+1}/{len(loader)} — loss: {out.loss.item():.4f}")
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labs, total_loss = [], [], 0.0
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += out.loss.item()
        preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
        labs.extend(labels.cpu().numpy())
    return total_loss/len(loader), f1_score(labs, preds, average='macro'), accuracy_score(labs, preds), preds, labs

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}
best_val_f1 = 0.0
epochs_no_improve = 0

print(f"🚀 Entraînement — jusqu'à {EPOCHS} epochs (patience={PATIENCE}) sur {DEVICE}\n")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"{'='*50}\n  EPOCH {epoch}/{EPOCHS}\n{'='*50}")
    t_epoch = time.time()

    train_loss = train_epoch(model, train_loader)
    val_loss, val_f1, val_acc, _, _ = evaluate(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)

    elapsed = (time.time() - t_epoch) / 60
    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val   loss : {val_loss:.4f}")
    print(f"  Val   F1   : {val_f1:.4f}  |  Acc : {val_acc:.4f}")
    print(f"  Durée      : {elapsed:.1f} min")

    # Checkpoint à chaque epoch — sécurité Studio (timeout, déconnexion, etc.)
    model.save_pretrained(CHECKPOINT_DIR / f"epoch_{epoch}")
    with open(WORKDIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        print(f"  ✅ Meilleur modèle sauvegardé (F1={best_val_f1:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  ⚠️  Pas d'amélioration ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"\n  🛑 Early stopping — arrêt à l'epoch {epoch}")
            break

    if epoch > 1 and val_loss > history['val_loss'][-2] and train_loss < history['train_loss'][-2]:
        print("  ⚠️  Signal d'overfitting détecté (train↓ mais val↑)")

total_time = (time.time() - t0) / 60
print(f"\n🏁 Entraînement terminé en {total_time:.1f} min — meilleur F1 val : {best_val_f1:.4f}")

🚀 Entraînement — jusqu'à 5 epochs (patience=2) sur cpu

  EPOCH 1/5
  Step 200/9133 — loss: 2.4594
  Step 400/9133 — loss: 2.2217
  Step 600/9133 — loss: 1.3402
  Step 800/9133 — loss: 1.4034
  Step 1000/9133 — loss: 1.6376
  Step 1200/9133 — loss: 1.4619
  Step 1400/9133 — loss: 0.7061
  Step 1600/9133 — loss: 0.9620
  Step 1800/9133 — loss: 0.9979
  Step 2000/9133 — loss: 0.9305
  Step 2200/9133 — loss: 0.6264
  Step 2400/9133 — loss: 0.5799
  Step 2600/9133 — loss: 1.1445
  Step 2800/9133 — loss: 1.2817
  Step 3000/9133 — loss: 0.7706
  Step 3200/9133 — loss: 1.4678
  Step 3400/9133 — loss: 1.0055
  Step 3600/9133 — loss: 1.3019
  Step 3800/9133 — loss: 1.2613
  Step 4000/9133 — loss: 0.6604
  Step 4200/9133 — loss: 0.7852
  Step 4400/9133 — loss: 1.1558
  Step 4600/9133 — loss: 0.5830
  Step 4800/9133 — loss: 0.7762
  Step 5000/9133 — loss: 1.3573
  Step 5200/9133 — loss: 1.3897
  Step 5400/9133 — loss: 1.0055
  Step 5600/9133 — loss: 1.2360
  Step 5800/9133 — loss: 0.8287
  Step 6

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ Meilleur modèle sauvegardé (F1=0.6976)
  EPOCH 2/5
  Step 200/9133 — loss: 0.3597
  Step 400/9133 — loss: 0.1732
  Step 600/9133 — loss: 1.0717
  Step 800/9133 — loss: 0.5755
  Step 1000/9133 — loss: 0.5576
  Step 1200/9133 — loss: 0.5171
  Step 1400/9133 — loss: 1.0060
  Step 1600/9133 — loss: 0.2959
  Step 1800/9133 — loss: 1.1949
  Step 2000/9133 — loss: 0.7687
  Step 2200/9133 — loss: 0.8069
  Step 2400/9133 — loss: 0.5559
  Step 2600/9133 — loss: 0.2455
  Step 2800/9133 — loss: 0.5349
  Step 3000/9133 — loss: 0.3505
  Step 3200/9133 — loss: 0.6620


KeyboardInterrupt: 

In [ ]:
# ── Cellule 8 : Évaluation finale sur TEST ───────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

best_model = DistilBertForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
test_loss, test_f1, test_acc, test_preds, test_labels = evaluate(best_model, test_loader)

print('\n' + '='*60)
print('  RÉSULTATS TEST — DistilBERT (meilleur checkpoint)')
print('='*60)
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))
print(f'  Test F1 macro  : {test_f1:.4f}')
print(f'  Test Accuracy  : {test_acc:.4f}')
print(f'  Baseline SVM   : 0.6515')
print(f'  Delta          : {test_f1 - 0.6515:+.4f} pts F1')
print('='*60)

cm = confusion_matrix(test_labels, test_preds, normalize='true')
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title(f'Matrice de confusion — DistilBERT (F1={test_f1:.4f})', fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(WORKDIR / 'distilbert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

n_epochs_run = len(history['train_loss'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, n_epochs_run + 1)
axes[0].plot(epochs_x, history['train_loss'], 'o-', color='#5B4FCF', lw=2, label='Train loss')
axes[0].plot(epochs_x, history['val_loss'],   'o-', color='#D85A30', lw=2, label='Val loss')
axes[0].set_title('Loss par epoch'); axes[0].legend()
axes[1].plot(epochs_x, history['val_f1'],  'o-', color='#1D9E75', lw=2, label='Val F1 macro')
axes[1].plot(epochs_x, history['val_acc'], 'o-', color='#5B4FCF', lw=2, linestyle='--', label='Val Accuracy')
axes[1].axhline(0.80, color='#D85A30', lw=1.5, linestyle=':', label='Objectif 0.80')
axes[1].set_title('Métriques par epoch'); axes[1].legend(); axes[1].set_ylim(0.5, 1.0)
plt.suptitle('DistilBERT — Courbes entraînement', fontweight='bold')
plt.tight_layout()
plt.savefig(WORKDIR / 'distilbert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cellule 9 : Sauvegarde des métriques ─────────────────────
metrics = {
    'model':          'distilbert-base-uncased',
    'platform':       'lightning-ai',
    'gpu':            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'epochs_run':     n_epochs_run,
    'epochs_max':     EPOCHS,
    'batch_size':     BATCH_SIZE,
    'learning_rate':  LEARNING_RATE,
    'weight_decay':   WEIGHT_DECAY,
    'dropout':        DROPOUT,
    'max_length':     MAX_LENGTH,
    'best_val_f1':    round(best_val_f1, 4),
    'test_f1_macro':  round(test_f1, 4),
    'test_accuracy':  round(test_acc, 4),
    'baseline_f1':    0.6515,
    'delta_f1':       round(test_f1 - 0.6515, 4),
    'num_labels':     NUM_LABELS,
    'class_names':    CLASS_NAMES,
    'history':        history,
}
with open(WORKDIR / 'training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('✅ Métriques sauvegardées →', WORKDIR / 'training_metrics.json')
print(f"\n   Tout est dans : {WORKDIR}")
print(f"   Ce dossier est PERSISTANT sur Lightning — pas besoin de télécharger")
print(f"   Tu peux y accéder via le terminal Lightning ou le panneau Files")

## Récupérer les résultats sur ta machine locale

Lightning AI Studio te donne accès à un **vrai terminal**. Ouvre-le (icône terminal dans le Studio) et lance :

```bash
# Voir ce qui a été généré
ls -la ~/newsops-artifacts/

# Option 1 — Zipper pour téléchargement via l'interface Lightning
cd ~ && zip -r newsops_distilbert.zip newsops-artifacts/
# Puis clic droit sur le fichier dans le panneau Files → Download

# Option 2 — Push direct vers ton repo GitHub depuis le Studio
# (Lightning a accès réseau complet, contrairement à certains sandbox)
git clone https://github.com/Dreipfelt/ai-newsops-platform.git
cp -r ~/newsops-artifacts/best_model ai-newsops-platform/models/distilbert/
cp ~/newsops-artifacts/training_metrics.json ai-newsops-platform/models/distilbert/
cp ~/newsops-artifacts/*.png ai-newsops-platform/reports/figures/
cd ai-newsops-platform
git add models/distilbert/ reports/figures/
git commit -m "feat: DistilBERT fine-tuned on Lightning AI (F1=$(python3 -c \"import json;print(json.load(open('models/distilbert/training_metrics.json'))['test_f1_macro'])\"))"
git push origin main
```

### Pourquoi ce notebook diffère de la version Colab

- Pas de `google.colab.files.upload/download` — Lightning a un vrai filesystem persistant accessible par terminal
- Le dossier `~/newsops-artifacts/` survit aux redémarrages du Studio (contrairement à `/content/` sur Colab qui est éphémère)
- Cache automatique du preprocessing : si tu relances le notebook, il ne re-télécharge pas le dataset
- `BATCH_SIZE` s'adapte automatiquement : 64 si GPU détecté, 16 si fallback CPU
- Accès `git` natif depuis le terminal du Studio — push direct possible sans dézipper localement